<a href="https://colab.research.google.com/github/jetbolima/2025_steam_dataset_analysis/blob/main/cleaned_Simple_transfomer_finetuning_and_creating_text_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jethro Elijah F. Bolima

This Google Colab notebook is heavily truncated vs my main project. I mostly use Jupyter Notebook but I only have 4 gb VRAM. I finished finetuning a transformer locally but I want to check out the other transformers as well. So, I am looking for free VRAM sources and this notebook is only for the finetuning and creation of text embeddings part of my main project.

In [ ]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
#import psycopg
#from pgvector.psycopg import register_vector
import os
import matplotlib.pyplot as plt
import gc
from joblib import dump, load

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Finetuning transformers for encoding

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss, BatchAllTripletLoss, BatchHardTripletLoss
from sentence_transformers.losses.BatchHardTripletLoss import BatchHardTripletLossDistanceFunction
from datasets import Dataset
import torch

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier # Faster than RF
from lightgbm import LGBMClassifier
import lightgbm as lgbm
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, PrecisionRecallDisplay, f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_extraction.text import CountVectorizer
#from bertopic.vectorizers import ClassTfidfTransformer
import scipy.sparse as sp
#import optuna
import shap

# SVC is avoided due to the high-cardinality of the dataset
# CategoricalNB is avoided due to floating datatype of the distances

In [ ]:
review_text_resampled_forEmbedding = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/review_text_resampled.parquet")

In [ ]:
review_text_resampled_forEmbedding = review_text_resampled_forEmbedding[["review_text", "voted_up"]]
review_text_resampled_forEmbedding["voted_up"] = review_text_resampled_forEmbedding["voted_up"].map({True : 1, False : 0})
review_text_resampled_forEmbedding.rename(columns = {"review_text" : "text", "voted_up" : "label"}, inplace = True)
review_text_resampled_forEmbedding.head()

,text,label
0,One of the best co-op experiences ever....\r\n...,1
1,"As a standalone chapter, its a bit short on re...",1
2,Cool little puzzle game!\nPortal is a small ga...,1
3,"Unfortunately it's a dead game, even on weeken...",1
4,"It was difficult at first, repeatedly switchin...",1


In [ ]:
train_idx = np.load("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/split_indices.npz")["train_idx"]
val_idx = np.load("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/split_indices.npz")["val_idx"]
test_idx = np.load("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/split_indices.npz")["test_idx"]

X_train = review_text_resampled_forEmbedding.iloc[train_idx]["text"].to_list()
X_val = review_text_resampled_forEmbedding.iloc[val_idx]["text"].to_list()
X_test = review_text_resampled_forEmbedding.iloc[test_idx]["text"].to_list()

In [ ]:
torch.cuda.is_available()

True

In [ ]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device = "cuda")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model_2 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device = "cuda")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### For First and Last Truncation


In [ ]:
tokenizer = model.tokenizer
tokenizer_2 = model_2.tokenizer

In [ ]:
def get_first_and_last(dataset, max_length, tokenizer):

    all_first_and_last_token_ids = []
    half_length = max_length // 2
    batch_size = 50000

    for i in range(0, len(dataset), batch_size):
        end = min(i + batch_size, len(dataset))
        batch = dataset[i : end]
        tokenized = tokenizer(batch, add_special_tokens = False, truncation = False, verbose = False)
        print(f"Finished tokenizing batch {i} - {end}!")

        first_and_last_batched = []
        for tokens in tokenized["input_ids"]:
            if len(tokens) <= max_length:
                first_and_last_batched.append(tokens)
            else:
                first = tokens[ : half_length]
                last = tokens[-half_length : ]
                first_and_last_batched.append(first + last)

        print(f"Finished concatenating first and last tokens!")

        first_and_last_list = tokenizer.batch_decode(first_and_last_batched, skip_special_tokens = True)
        print(f"Finished decoding batch {i} - {end}! \n")

        all_first_and_last_token_ids.extend(first_and_last_list)

    return all_first_and_last_token_ids

In [ ]:
X_train_embeddings =  get_first_and_last(X_train, 512, tokenizer)
X_val_embeddings =  get_first_and_last(X_val, 512, tokenizer)
X_test_embeddings =  get_first_and_last(X_test, 512, tokenizer)

Finished tokenizing batch 0 - 50000!
Finished concatenating first and last tokens!
Finished decoding batch 0 - 50000! 

Finished tokenizing batch 50000 - 100000!
Finished concatenating first and last tokens!
Finished decoding batch 50000 - 100000! 

Finished tokenizing batch 100000 - 150000!
Finished concatenating first and last tokens!
Finished decoding batch 100000 - 150000! 

Finished tokenizing batch 150000 - 200000!
Finished concatenating first and last tokens!
Finished decoding batch 150000 - 200000! 

Finished tokenizing batch 200000 - 250000!
Finished concatenating first and last tokens!
Finished decoding batch 200000 - 250000! 

Finished tokenizing batch 250000 - 300000!
Finished concatenating first and last tokens!
Finished decoding batch 250000 - 300000! 

Finished tokenizing batch 300000 - 350000!
Finished concatenating first and last tokens!
Finished decoding batch 300000 - 350000! 

Finished tokenizing batch 350000 - 400000!
Finished concatenating first and last tokens!
F

In [ ]:
X_train_embeddings_2 =  get_first_and_last(X_train, 256, tokenizer_2)
X_val_embeddings_2 =  get_first_and_last(X_val, 256, tokenizer_2)
X_test_embeddings_2 =  get_first_and_last(X_test, 256, tokenizer_2)

Finished tokenizing batch 0 - 50000!
Finished concatenating first and last tokens!
Finished decoding batch 0 - 50000! 

Finished tokenizing batch 50000 - 100000!
Finished concatenating first and last tokens!
Finished decoding batch 50000 - 100000! 

Finished tokenizing batch 100000 - 150000!
Finished concatenating first and last tokens!
Finished decoding batch 100000 - 150000! 

Finished tokenizing batch 150000 - 200000!
Finished concatenating first and last tokens!
Finished decoding batch 150000 - 200000! 

Finished tokenizing batch 200000 - 250000!
Finished concatenating first and last tokens!
Finished decoding batch 200000 - 250000! 

Finished tokenizing batch 250000 - 300000!
Finished concatenating first and last tokens!
Finished decoding batch 250000 - 300000! 

Finished tokenizing batch 300000 - 350000!
Finished concatenating first and last tokens!
Finished decoding batch 300000 - 350000! 

Finished tokenizing batch 350000 - 400000!
Finished concatenating first and last tokens!
F

In [ ]:
X_train_embeddings = pd.DataFrame({"text" : X_train_embeddings})
X_val_embeddings = pd.DataFrame({"text" : X_val_embeddings})
X_test_embeddings = pd.DataFrame({"text" : X_test_embeddings})

X_train_embeddings.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_train_embeddings.parquet")
X_val_embeddings.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_val_embeddings.parquet")
X_test_embeddings.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_test_embeddings.parquet")

In [ ]:
X_train_embeddings_2 = pd.DataFrame({"text" : X_train_embeddings_2})
X_val_embeddings_2 = pd.DataFrame({"text" : X_val_embeddings_2})
X_test_embeddings_2 = pd.DataFrame({"text" : X_test_embeddings_2})

X_train_embeddings_2.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_train_embeddings_2.parquet")
X_val_embeddings_2.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_val_embeddings_2.parquet")
X_test_embeddings_2.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_test_embeddings_2.parquet")

ValueError: If using all scalar values, you must pass an index

In [ ]:
X_train_embeddings = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_train_embeddings.parquet")

In [ ]:
X_train_finetune = pd.DataFrame({
    "text": X_train_embeddings["text"].to_list(),
    "label": review_text_resampled_forEmbedding.iloc[train_idx]["label"].to_list()
})

In [ ]:
X_train_embeddings_2 = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_train_embeddings_2.parquet")

In [ ]:
X_train_finetune_2 = pd.DataFrame({
    "text": X_train_embeddings_2["text"].to_list(),
    "label": review_text_resampled_forEmbedding.iloc[train_idx]["label"].to_list()
})

### For BatchedAllTriples

In [ ]:
X_train_finetune

,text,label
0,"i'm sorry, but i didn't like this game. i real...",0
1,game hunters fr cooked on this,1
2,i found this vn extremely boring. there was ba...,0
3,you're not alone if you thought batds was shal...,1
4,a very strange / unique 3d rts i've remembered...,1
...,...,...
435790,this game is super hot,1
435791,i would recommend this game to staunch fans of...,1
435792,the campaign's replay value is completely dest...,0
435793,"no updates, no guides. fairly simple but even ...",0


In [ ]:
X_train_finetune.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/X_train_finetune.parquet")

In [ ]:
X_train_finetune = Dataset.from_pandas(X_train_finetune, preserve_index = False)
X_train_finetune.save_to_disk("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/")

Saving the dataset (0/1 shards):   0%|          | 0/435795 [00:00<?, ? examples/s]

In [ ]:
X_train_finetune_2

,text,label
0,"i ' m sorry, but i didn ' t like this game. i ...",0
1,game hunters fr cooked on this,1
2,i found this vn extremely boring. there was ba...,0
3,you ' re not alone if you thought batds was sh...,1
4,a very strange / unique 3d rts i ' ve remember...,1
...,...,...
435790,this game is super hot,1
435791,i would recommend this game to staunch fans of...,1
435792,the campaign ' s replay value is completely de...,0
435793,"no updates, no guides. fairly simple but even ...",0


In [ ]:
X_train_finetune_2.to_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/X_train_finetune_2.parquet")

In [ ]:
X_train_finetune_2 = Dataset.from_pandas(X_train_finetune_2, preserve_index = False)
X_train_finetune_2.save_to_disk("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/miniLM-L6/")

Saving the dataset (0/1 shards):   0%|          | 0/435795 [00:00<?, ? examples/s]

### Start of Finetuning

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
max_steps = 10215
remaining_steps = 4983

In [ ]:
X_train_finetune = Dataset.load_from_disk("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/")

In [ ]:
model = SentenceTransformer("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/training/continued_training/checkpoint-6000", device = "cuda")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
loss = BatchAllTripletLoss(model, distance_metric = BatchHardTripletLossDistanceFunction.cosine_distance, margin = 0.5)
#loss = BatchHardTripletLoss(model, distance_metric = BatchHardTripletLossDistanceFunction.cosine_distance, margin = 0.7)

In [ ]:
training_args = SentenceTransformerTrainingArguments(
        output_dir = "/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/training/continued_training",
        num_train_epochs = 3,
        #max_steps = max_steps - remaining_steps,
        per_device_train_batch_size = 64,
        gradient_accumulation_steps = 2,
        learning_rate = 2e-5 ,# default is 2e-5
        lr_scheduler_type = "cosine",
        warmup_steps = 0.1,
        weight_decay = 0.01, # adds L2 regularization, penalize large weights
        fp16 = True,
        logging_steps = 100,
        save_steps = 1000,
        save_total_limit= 1
    )

trainer = SentenceTransformerTrainer(
        model = model,
        args = training_args,
        train_dataset = X_train_finetune,
        loss = loss
    )

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
trainer.train(resume_from_checkpoint = True)
print("Finished training!")

model.save_pretrained("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/BGE-small-finetuned-MNR/Finished Model - Continued/")
print("Model is saved!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Step,Training Loss
6100,0.702764
6200,0.702696
6300,0.702672
6400,0.702620
6500,0.702701
6600,0.702551
6700,0.702572
6800,0.702553
6900,0.702150
7000,0.702448


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Finished training!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model is saved!


In [ ]:
trainer.save_model("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/batchalltriple/training/mid_checkpoint")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Check Embeddings

In [ ]:
X_val_embeddings = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_val_embeddings.parquet")
X_val_embeddings_2 = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_val_embeddings_2.parquet")
X_val_embeddings_baseline = pd.DataFrame({
    "text": X_val,
    "label": review_text_resampled_forEmbedding.iloc[val_idx]["label"].to_list()
})

In [ ]:
X_val_finetune_0 = X_val_embeddings_baseline[X_val_embeddings_baseline["label"] == 0].reset_index(drop = True)
X_val_finetune_1 = X_val_embeddings_baseline[X_val_embeddings_baseline["label"] == 1].reset_index(drop = True)

In [ ]:
X_val_finetune_1 = X_val_finetune_1.sample(frac = 1, random_state = 8)
X_val_finetune_0 = X_val_finetune_0.sample(frac = 1, random_state = 8)

X_val_finetune_0_pos = X_val_finetune_0.iloc[:len(X_val_finetune_0)//2]
X_val_finetune_0_neg = X_val_finetune_0.iloc[-len(X_val_finetune_0)//2:]

In [ ]:
X_val_finetune_1_pos = X_val_finetune_1.iloc[:len(X_val_finetune_0_pos)]
X_val_finetune_1_neg = X_val_finetune_1.iloc[-len(X_val_finetune_0_neg):]

In [ ]:
X_val_finetune_1_pos1 = X_val_finetune_1_pos[:len(X_val_finetune_1_pos)//2].reset_index(drop = True)
X_val_finetune_1_pos2 = X_val_finetune_1_pos[-len(X_val_finetune_1_pos1):].reset_index(drop = True)

In [ ]:
X_val_finetune_1_pos_final = X_val_finetune_1_pos1.copy()
X_val_finetune_1_pos_final["text2"] =  X_val_finetune_1_pos2["text"]
X_val_finetune_1_pos_final.drop(columns = "label", inplace = True)
X_val_finetune_1_pos_final["label"] = 1

In [ ]:
X_val_finetune_0_pos1 = X_val_finetune_0_pos[:len(X_val_finetune_0_pos)//2].reset_index(drop = True)
X_val_finetune_0_pos2 = X_val_finetune_0_pos[-len(X_val_finetune_0_pos1):].reset_index(drop = True)

In [ ]:
X_val_finetune_0_pos_final = X_val_finetune_0_pos1.copy()
X_val_finetune_0_pos_final["text2"] =  X_val_finetune_0_pos2["text"]
X_val_finetune_0_pos_final.drop(columns = "label", inplace = True)
X_val_finetune_0_pos_final["label"] = 1

In [ ]:
X_val_finetune_0_neg_semifinal = X_val_finetune_0_neg.iloc[:len(X_val_finetune_0_pos_final)+len(X_val_finetune_1_pos_final)].reset_index(drop = True)
X_val_finetune_1_neg_semifinal = X_val_finetune_1_neg.iloc[:len(X_val_finetune_0_neg_semifinal)].reset_index(drop = True)

In [ ]:
X_val_finetune_neg_final = X_val_finetune_0_neg_semifinal.copy()
X_val_finetune_neg_final["text2"] = X_val_finetune_1_neg_semifinal["text"]
X_val_finetune_neg_final.drop(columns = "label", inplace = True)
X_val_finetune_neg_final["label"] = 0

In [ ]:
X_val_finetune_baseline = pd.concat([X_val_finetune_1_pos_final,
                                           X_val_finetune_0_pos_final,
                                           X_val_finetune_neg_final])

X_val_finetune_baseline.rename(columns = {"text" : "text_1", "text2" : "text_2"}, inplace = True)
X_val_finetune_baseline["label"] = X_val_finetune_baseline["label"].astype("int")
X_val_finetune_baseline.head()

,text_1,text_2,label
0,"Listen, I've played them all, yes, all of them...","Very fun car combat game, not enough of those ...",1
1,"roulette-like games is boring for me, but you ...","Very goofy game, but also very entertaining. T...",1
2,"Good game,\nTiger is cool!\nAll I'll say is wa...",The Invincible is one of the most immersive na...,1
3,I loved this game - and I rarely play video ga...,"game keeps on maturing with each patch/dlc, ke...",1
4,Nice art game,"I don't get the negative reviews, tbh. If you ...",1


In [ ]:
model = SentenceTransformer("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/BGE-small-finetuned-MNR/Finished Model - Continued/", device = "cuda")
baseline_model = SentenceTransformer("BAAI/bge-m3", device = "cuda")
model_alter = SentenceTransformer("/content/drive/MyDrive/For_Colab_GPU_use/1. a. Cleaner Containers/minilm-l6-finetuned-MNR/Finished Model/", device = "cuda")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
X_val_finetune = pd.DataFrame({
    "text": X_val_embeddings["text"].to_list(),
    "label": review_text_resampled_forEmbedding.iloc[val_idx]["label"].to_list()
})

In [ ]:
X_val_finetune_0 = X_val_finetune[X_val_finetune["label"] == 0].reset_index(drop = True)
X_val_finetune_1 = X_val_finetune[X_val_finetune["label"] == 1].reset_index(drop = True)

In [ ]:
X_val_finetune_1 = X_val_finetune_1.sample(frac = 1, random_state = 8)
X_val_finetune_0 = X_val_finetune_0.sample(frac = 1, random_state = 8)

X_val_finetune_0_pos = X_val_finetune_0.iloc[:len(X_val_finetune_0)//2]
X_val_finetune_0_neg = X_val_finetune_0.iloc[-len(X_val_finetune_0)//2:]

In [ ]:
X_val_finetune_1_pos = X_val_finetune_1.iloc[:len(X_val_finetune_0_pos)]
X_val_finetune_1_neg = X_val_finetune_1.iloc[-len(X_val_finetune_0_neg):]

In [ ]:
X_val_finetune_1_pos1 = X_val_finetune_1_pos[:len(X_val_finetune_1_pos)//2].reset_index(drop = True)
X_val_finetune_1_pos2 = X_val_finetune_1_pos[-len(X_val_finetune_1_pos1):].reset_index(drop = True)

In [ ]:
X_val_finetune_1_pos_final = X_val_finetune_1_pos1.copy()
X_val_finetune_1_pos_final["text2"] =  X_val_finetune_1_pos2["text"]
X_val_finetune_1_pos_final.drop(columns = "label", inplace = True)
X_val_finetune_1_pos_final["label"] = 1

In [ ]:
X_val_finetune_0_pos1 = X_val_finetune_0_pos[:len(X_val_finetune_0_pos)//2].reset_index(drop = True)
X_val_finetune_0_pos2 = X_val_finetune_0_pos[-len(X_val_finetune_0_pos1):].reset_index(drop = True)

In [ ]:
X_val_finetune_0_pos_final = X_val_finetune_0_pos1.copy()
X_val_finetune_0_pos_final["text2"] =  X_val_finetune_0_pos2["text"]
X_val_finetune_0_pos_final.drop(columns = "label", inplace = True)
X_val_finetune_0_pos_final["label"] = 1

In [ ]:
X_val_finetune_0_neg_semifinal = X_val_finetune_0_neg.iloc[:len(X_val_finetune_0_pos_final)+len(X_val_finetune_1_pos_final)].reset_index(drop = True)
X_val_finetune_1_neg_semifinal = X_val_finetune_1_neg.iloc[:len(X_val_finetune_0_neg_semifinal)].reset_index(drop = True)

In [ ]:
X_val_finetune_neg_final = X_val_finetune_0_neg_semifinal.copy()
X_val_finetune_neg_final["text2"] = X_val_finetune_1_neg_semifinal["text"]
X_val_finetune_neg_final.drop(columns = "label", inplace = True)
X_val_finetune_neg_final["label"] = 0

In [ ]:
X_val_finetune_eval = pd.concat([X_val_finetune_1_pos_final,
                                           X_val_finetune_0_pos_final,
                                           X_val_finetune_neg_final])

X_val_finetune_eval.rename(columns = {"text" : "text_1", "text2" : "text_2"}, inplace = True)
X_val_finetune_eval["label"] = X_val_finetune_eval["label"].astype("int")
X_val_finetune_eval.head()

,text_1,text_2,label
0,"listen, i've played them all, yes, all of them...","very fun car combat game, not enough of those ...",1
1,"roulette - like games is boring for me, but yo...","very goofy game, but also very entertaining. t...",1
2,"good game, tiger is cool! all i'll say is watc...",the invincible is one of the most immersive na...,1
3,i loved this game - and i rarely play video ga...,"game keeps on maturing with each patch / dlc, ...",1
4,nice art game,"i don't get the negative reviews, tbh. if you ...",1


In [ ]:
X_val_finetune = pd.DataFrame({
    "text": X_val_embeddings_2["text"].to_list(),
    "label": review_text_resampled_forEmbedding.iloc[val_idx]["label"].to_list()
})

In [ ]:
X_val_finetune_0 = X_val_finetune[X_val_finetune["label"] == 0].reset_index(drop = True)
X_val_finetune_1 = X_val_finetune[X_val_finetune["label"] == 1].reset_index(drop = True)

In [ ]:
X_val_finetune_1 = X_val_finetune_1.sample(frac = 1, random_state = 8)
X_val_finetune_0 = X_val_finetune_0.sample(frac = 1, random_state = 8)

X_val_finetune_0_pos = X_val_finetune_0.iloc[:len(X_val_finetune_0)//2]
X_val_finetune_0_neg = X_val_finetune_0.iloc[-len(X_val_finetune_0)//2:]

In [ ]:
X_val_finetune_1_pos = X_val_finetune_1.iloc[:len(X_val_finetune_0_pos)]
X_val_finetune_1_neg = X_val_finetune_1.iloc[-len(X_val_finetune_0_neg):]

In [ ]:
X_val_finetune_1_pos1 = X_val_finetune_1_pos[:len(X_val_finetune_1_pos)//2].reset_index(drop = True)
X_val_finetune_1_pos2 = X_val_finetune_1_pos[-len(X_val_finetune_1_pos1):].reset_index(drop = True)

In [ ]:
X_val_finetune_1_pos_final = X_val_finetune_1_pos1.copy()
X_val_finetune_1_pos_final["text2"] =  X_val_finetune_1_pos2["text"]
X_val_finetune_1_pos_final.drop(columns = "label", inplace = True)
X_val_finetune_1_pos_final["label"] = 1

In [ ]:
X_val_finetune_0_pos1 = X_val_finetune_0_pos[:len(X_val_finetune_0_pos)//2].reset_index(drop = True)
X_val_finetune_0_pos2 = X_val_finetune_0_pos[-len(X_val_finetune_0_pos1):].reset_index(drop = True)

In [ ]:
X_val_finetune_0_pos_final = X_val_finetune_0_pos1.copy()
X_val_finetune_0_pos_final["text2"] =  X_val_finetune_0_pos2["text"]
X_val_finetune_0_pos_final.drop(columns = "label", inplace = True)
X_val_finetune_0_pos_final["label"] = 1

In [ ]:
X_val_finetune_0_neg_semifinal = X_val_finetune_0_neg.iloc[:len(X_val_finetune_0_pos_final)+len(X_val_finetune_1_pos_final)].reset_index(drop = True)
X_val_finetune_1_neg_semifinal = X_val_finetune_1_neg.iloc[:len(X_val_finetune_0_neg_semifinal)].reset_index(drop = True)

In [ ]:
X_val_finetune_neg_final = X_val_finetune_0_neg_semifinal.copy()
X_val_finetune_neg_final["text2"] = X_val_finetune_1_neg_semifinal["text"]
X_val_finetune_neg_final.drop(columns = "label", inplace = True)
X_val_finetune_neg_final["label"] = 0

In [ ]:
X_val_finetune_eval_2 = pd.concat([X_val_finetune_1_pos_final,
                                           X_val_finetune_0_pos_final,
                                           X_val_finetune_neg_final])

X_val_finetune_eval_2.rename(columns = {"text" : "text_1", "text2" : "text_2"}, inplace = True)
X_val_finetune_eval_2["label"] = X_val_finetune_eval_2["label"].astype("int")
X_val_finetune_eval_2.head()

,text_1,text_2,label
0,"listen, i ' ve played them all, yes, all of th...","very fun car combat game, not enough of those ...",1
1,"roulette - like games is boring for me, but yo...","very goofy game, but also very entertaining. t...",1
2,"good game, tiger is cool! all i ' ll say is wa...",the invincible is one of the most immersive na...,1
3,i loved this game - and i rarely play video ga...,"game keeps on maturing with each patch / dlc, ...",1
4,nice art game,"i don ' t get the negative reviews, tbh. if yo...",1


In [ ]:
from sentence_transformers.evaluation import BinaryClassificationEvaluator

In [ ]:
evaluator = BinaryClassificationEvaluator(
    sentences1 =  X_val_finetune_eval["text_1"].to_list(),
    sentences2 =  X_val_finetune_eval["text_2"].to_list(),
    labels = X_val_finetune_eval["label"].to_list(),
    show_progress_bar = True,
    batch_size = 128,
    name = "steam_bge_small_embedding_finetune_eval")

evaluator_2 = BinaryClassificationEvaluator(
    sentences1 =  X_val_finetune_eval_2["text_1"].to_list(),
    sentences2 =  X_val_finetune_eval_2["text_2"].to_list(),
    labels = X_val_finetune_eval_2["label"].to_list(),
    show_progress_bar = True,
    batch_size = 128,
    name = "steam_miniLM_L6_finetune_eval")


evaluator_baseline = BinaryClassificationEvaluator(
    sentences1 =  X_val_finetune_baseline["text_1"].to_list(),
    sentences2 =  X_val_finetune_baseline["text_2"].to_list(),
    labels = X_val_finetune_baseline["label"].to_list(),
    show_progress_bar = True,
    batch_size = 32,
    name = "steam_bge_m3_baseline_finetune_eval")

In [ ]:
score_model = evaluator(model)
score_baseline = evaluator_baseline(baseline_model)
score_alter = evaluator_2(model_alter)

print(f"Baseline BGE-M3 Score: {score_baseline}")
print(f"Finetuned BGE-small Score: {score_model}")
print(f"Finetuned MiniLM-L6 Score: {score_alter}")


Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Batches:   0%|          | 0/822 [00:00<?, ?it/s]

Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Baseline BGE-M3 Score: {'steam_bge_m3_baseline_finetune_eval_cosine_accuracy': 0.6330185851318945, 'steam_bge_m3_baseline_finetune_eval_cosine_accuracy_threshold': 0.5307102203369141, 'steam_bge_m3_baseline_finetune_eval_cosine_f1': 0.6696157330917231, 'steam_bge_m3_baseline_finetune_eval_cosine_f1_threshold': 0.38268423080444336, 'steam_bge_m3_baseline_finetune_eval_cosine_precision': 0.5195600033082458, 'steam_bge_m3_baseline_finetune_eval_cosine_recall': 0.9415467625899281, 'steam_bge_m3_baseline_finetune_eval_cosine_ap': 0.6784812565004921, 'steam_bge_m3_baseline_finetune_eval_cosine_mcc': 0.1215218842006049}
Finetuned BGE-small Score: {'steam_bge_small_embedding_finetune_eval_cosine_accuracy': 0.8813699040767387, 'steam_bge_small_embedding_finetune_eval_cosine_accuracy_threshold': 0.9976645708084106, 'steam_bge_small_embedding_finetune_eval_cosine_f1': 0.8813508667065153, 'steam_bge_small_embedding_finetune_eval_cosine_f1_threshold': 0.9975886344909668, 'steam_bge_small_embedding_

### Encoding the embeddings

In [ ]:
X_train_embeddings = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_train_embeddings.parquet")
X_val_embeddings = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_val_embeddings.parquet")
X_test_embeddings = pd.read_parquet("/content/drive/MyDrive/For_Colab_GPU_use/1. SentenceTransformer - BGE-small/X_test_embeddings.parquet")

X_train_embeddings = X_train_embeddings["text"].to_list()
X_val_embeddings = X_val_embeddings["text"].to_list()
X_test_embeddings = X_test_embeddings["text"].to_list()

In [ ]:
X_train_embeddings = model.encode(X_train_embeddings, batch_size = 512, show_progress_bar = True, normalize_embeddings = True, convert_to_numpy = True)
np.save("/content/drive/MyDrive/embeddings/BGE_small_finetune/X_train_embeddings_alter.npy", X_train_embeddings)

X_val_embeddings = model.encode(X_val_embeddings, batch_size = 512, show_progress_bar = True, normalize_embeddings = True, convert_to_numpy = True)
np.save("/content/drive/MyDrive/embeddings/BGE_small_finetune/X_val_embeddings_alter.npy", X_val_embeddings)

X_test_embeddings = model.encode(X_test_embeddings, batch_size = 512, show_progress_bar = True, normalize_embeddings = True, convert_to_numpy = True)
np.save("/content/drive/MyDrive/embeddings/BGE_small_finetune/X_test_embeddings_alter.npy", X_test_embeddings)

Batches:   0%|          | 0/852 [00:00<?, ?it/s]

Batches:   0%|          | 0/107 [00:00<?, ?it/s]

Batches:   0%|          | 0/107 [00:00<?, ?it/s]